# Twin4Build canonical estimation benchmark — Colab runner

Runs the **CUDA estimation scaling matrix** (single shooting with SLSQP, the custom batched SQP with 1 and 8 starts,
and IPOPT collocation) on the canonical `full_workflow` model for 1 / 10 / 50 / 100 zones, 120 hours, 360 × 20-minute steps.

Why Colab: on the 8 GB laptop GPU the 50-zone shooting bundle commits ~10.8 GB and pages over PCIe, and collocation above
one zone was never measured. Pick a runtime with a large GPU (**A100 40/80 GB** or L4 24 GB; Runtime ▸ Change runtime type).

How it works
* Each case runs in a **fresh subprocess** and is checkpointed to **Google Drive** as soon as it finishes, so a session
  timeout costs at most the case in flight. Re-running the notebook resumes from the checkpoint.
* Timings are *active* seconds (a frozen process is subtracted); every row carries peak VRAM / RAM and a
  `cuda_oversubscribed` flag.
* Results from this GPU go to their own Drive folder — do not merge them with laptop rows in one scaling table.

Twin4Build is installed from the **active branch** on GitHub, exactly like the earlier benchmark notebooks
(`git clone` + `checkout GIT_REF` + `pip install -e`). The branch must contain the boundary-state box fix (issue #133).

In [ ]:
#@title 1. Configuration { display-mode: "form" }
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"  #@param {type:"string"}
GIT_REF = "feature/issue-128/collocation-initialization"       #@param {type:"string"}
RESULTS_IN_DRIVE = "Twin4Build_benchmarks/results_colab"   #@param {type:"string"}

ZONES = [1, 10, 50, 100]          #@param {type:"raw"}
ARMS = ["slsqp-single-shooting", "custom-batched-sqp:1", "custom-batched-sqp:8", "ipopt-collocation"]  #@param {type:"raw"}
INCLUDE_HYBRID_ARM = False        #@param {type:"boolean"}
RUN_CPU_ARM = False               #@param {type:"boolean"}
SOLVER_BUDGET = 300               #@param {type:"integer"}
SMOKE_TEST_FIRST = True           #@param {type:"boolean"}

In [ ]:
#@title 2. GPU check and Drive mount
import subprocess, os, sys, shutil, json, time
from pathlib import Path
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
RESULTS_DIR = DRIVE / RESULTS_IN_DRIVE
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("results dir:", RESULTS_DIR)

In [ ]:
#@title 3. Install Twin4Build from the active branch (same bootstrap as the earlier benchmark notebooks)
import pathlib
REPO = pathlib.Path("/content/Twin4Build")
if "google.colab" in sys.modules:
    if not REPO.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False, capture_output=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "psutil"], check=True)
else:
    REPO = pathlib.Path.cwd()
    if REPO.name == "benchmarks":
        REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("installed", GIT_REF, "@", commit)
assert (REPO / "twin4build" / "estimator" / "_collocation.py").exists(), "this ref predates the collocation fix (issue #133)"
import torch, casadi, twin4build
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("casadi", casadi.__version__, "| twin4build", twin4build.__file__)

In [ ]:
#@title 4. Point the harness' results directory at Drive
local_results = REPO / "generated_files" / "canonical_benchmarks"
local_results.parent.mkdir(exist_ok=True)
if local_results.is_symlink() or local_results.exists():
    if local_results.is_symlink():
        local_results.unlink()
    else:
        shutil.rmtree(local_results)
local_results.symlink_to(RESULTS_DIR, target_is_directory=True)
print(local_results, "->", os.readlink(local_results))
ckpt = RESULTS_DIR / "estimation_scaling_in_progress.json"
if ckpt.exists():
    rows = json.loads(ckpt.read_text())["rows"]
    print(f"existing checkpoint with {len(rows)} rows -> this run resumes")
else:
    print("no checkpoint yet -> fresh run")

In [ ]:
#@title 5. Configure the matrix for this GPU
os.environ["T4B_BENCHMARK_MODE"] = "full"
os.environ.pop("T4B_BENCHMARK_DEFER_SOLVERS", None)
sys.path.insert(0, str(REPO))
import importlib, benchmarks.common as common
importlib.reload(common)

matrix = []
for arm in ARMS:
    solver, _, starts = arm.partition(":")
    matrix.append(("cuda", solver, int(starts or 1)))
if INCLUDE_HYBRID_ARM:
    matrix.append(("cuda", "slsqp5-ipopt-collocation", 1))
if RUN_CPU_ARM:
    matrix.insert(0, ("cpu", "slsqp-single-shooting", 1))
common.ESTIMATION_MATRIX[:] = matrix
common.ZONE_COUNTS[:] = list(ZONES)
common.ESTIMATION_CPU_MAX_ZONES = 10 if RUN_CPU_ARM else 0
common.ESTIMATION_SOLVER_BUDGET = int(SOLVER_BUDGET)

config = common.BenchmarkConfig(mode="full")
print("zones", config.zone_counts, "| budget", config.estimation_maxiter, "| hours", config.hours)
for n in config.zone_counts:
    for dev, solver, ns in common.ESTIMATION_MATRIX:
        pf = common._collocation_preflight(n, config.hours) if solver in common.COLLOCATION_SOLVERS else {}
        extra = f"  nlp vars {pf['n_collocation_nlp_variables']:,}" if pf else ""
        print(f"  {n:3d} zones  {dev:4s} {solver:26s} starts={ns}{extra}")

In [ ]:
#@title 6. Optional smoke test (2-hour horizon, 1 iteration per case; validates the environment)
if SMOKE_TEST_FIRST:
    smoke = common.BenchmarkConfig(mode="smoke")
    t0 = time.time()
    smoke_rows = common.run_estimation_scaling(smoke)
    print(f"smoke: {len(smoke_rows)} rows in {time.time()-t0:.0f} s")
    for r in smoke_rows:
        print("  ", r.get("n_zones"), r.get("device"), r.get("solver"), r.get("n_starts"), r.get("status"), r.get("reason") or r.get("error") or "")
    bad = [r for r in smoke_rows if r.get("status") == "failed"]
    assert not bad, "smoke test failed; see rows above"

In [ ]:
#@title 7. Run (resumable — re-run this cell after a session timeout)
common.seed_everything(config.seed)
t0 = time.time()
rows = common.run_estimation_scaling(config)
path = common.serialize_results("estimation_scaling", config, rows)
print(f"done in {(time.time()-t0)/3600:.2f} h -> {path}")
from collections import Counter
print(dict(Counter(r.get("status") for r in rows)))

In [ ]:
#@title 8. Results table
import pandas as pd
rows = json.loads((RESULTS_DIR / "estimation_scaling_in_progress.json").read_text())["rows"]
def q50(r, sig):
    try: return round(r["prediction_quality"][sig]["per_zone_rmse_quantiles"]["q50"], 4)
    except Exception: return None
table = pd.DataFrame([{
    "zones": r.get("n_zones"), "device": r.get("device"), "solver": r.get("solver"), "starts": r.get("n_starts"),
    "status": r.get("status"), "iters": r.get("iterations"), "active_s": round(r.get("seconds") or 0, 1),
    "wall_s": round(r.get("wall_seconds") or 0, 1), "frozen_s": round(r.get("frozen_seconds") or 0, 1),
    "temp_rmse": q50(r, "temperature"), "valve_rmse": q50(r, "valve"), "damper_rmse": q50(r, "damper"), "co2_rmse": q50(r, "co2"),
    "peak_vram_GiB": round(((r.get("memory") or {}).get("cuda_device_used_peak_bytes") or 0) / 2**30, 2),
    "torch_reserved_GiB": round(((r.get("memory") or {}).get("torch_cuda_max_reserved_bytes") or 0) / 2**30, 2),
    "oversubscribed": (r.get("memory") or {}).get("cuda_oversubscribed"),
    "peak_ram_GiB": round(((r.get("memory") or {}).get("host_peak_working_set_bytes") or 0) / 2**30, 1),
    "message": str(r.get("message") or r.get("reason") or r.get("error") or "")[:60],
} for r in rows if r.get("status") != "skipped"]).sort_values(["zones", "solver", "starts"])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
display(table)
env = json.loads((RESULTS_DIR / "estimation_scaling_in_progress.json").read_text()).get("environment", {})
print({k: env.get(k) for k in ("cuda_device", "torch", "cuda_runtime", "git_commit", "platform")})

### Notes
* `fit_objective` is on different scales for shooting (rescaled to 100 at x0) and collocation (absolute mean weighted
  squared residual); compare arms on the RMSE columns, which are on the noise floor at 0.05 °C / 0.025 / 0.025 / 15 ppm.
* Collocation rows run to the iteration cap unless early stopping fires on feasible stagnation; their tail is slow on
  flat parameter directions, so "not converged (iteration cap)" with noise-floor RMSEs is the expected outcome.
* To add repetitions or other arms, edit cell 1 and re-run from cell 5; completed cases are never re-run.
* Copy `estimation_scaling_in_progress.json` from the Drive results folder back to the laptop to fold these rows into the canvas.